# Raw Positions — Timemajor Tracking Data

Loads JSONL tracking data from `s3://bundesliga-2022-2023-data/timemajor/` and saves as a Delta table in `bundesliga-2022-2023.batch.raw_positions`.

Each JSONL file is one match. Each line is one frame (timestamp) containing an array of all entities (players, ball, referees) with their X/Y/Z positions, speed, and ball possession status.

## Raw Positions Transformation Steps

Source: `s3://bundesliga-2022-2023-data/timemajor/*.jsonl` (7 files, one per match) → Output: `batch.raw_positions` (15 columns, \~23.7M rows)

| Step | Cell | Action | Columns / Result |
| --- | --- | --- | --- |
| 0 | 3 | Read all JSONL files from S3 via `spark.read.json()` | `raw_df`: 2 top-level columns (`frames` array, `n` line number). Each line = one frame with an array of entities. |
| 1 | 4 | Explode `frames` array and flatten into one row per entity per frame. Cast all string fields to proper types (long, double, timestamp, int). | `positions_df`: 15 columns — `frame_id`, `match_id`, `game_section`, `team_id`, `person_id`, `timestamp`, `x`, `y`, `z`, `speed`, `distance`, `acceleration`, `m_flag`, `ball_possession`, `ball_status` |
| 2 | 5 | Save `positions_df` as Delta table `batch.raw_positions` (overwrite mode). | 23,739,462 rows, 15 columns, 7 matches. Partitioned by `match_id` in downstream use. |

**Output schema (15 columns):**

| # | Column | Type | Source Field | Description |
| --- | --- | --- | --- | --- |
| 1 | `frame_id` | `bigint` | `n` | Sequential frame number within a match. Primary frame identifier (25 FPS). |
| 2 | `match_id` | `string` | `FrameSet.MatchId` | Unique match identifier (e.g. `DFL-MAT-J03WMX`). FK to `match_info`. |
| 3 | `game_section` | `string` | `FrameSet.GameSection` | `firstHalf` or `secondHalf`. Teams switch sides at halftime. |
| 4 | `team_id` | `string` | `FrameSet.TeamId` | Home team ID, guest team ID, `BALL`, or `referee`. |
| 5 | `person_id` | `string` | `FrameSet.PersonId` | Unique entity identifier (e.g. `DFL-OBJ-0002HE`). `BALL` for ball rows. |
| 6 | `timestamp` | `timestamp` | `Frame.T` | Wall-clock time of the frame. |
| 7 | `x` | `double` | `Frame.X` | Pitch X in meters. **Absolute**: \~-52.5 to +52.5, center at 0. Goal-to-goal axis. |
| 8 | `y` | `double` | `Frame.Y` | Pitch Y in meters. \~-34 to +34, center at 0. Sideline-to-sideline axis. |
| 9 | `z` | `double` | `Frame.Z` | Height in meters (ball height, player jumps). Null for most rows. |
| 10 | `speed` | `double` | `Frame.S` | Entity speed in m/s. |
| 11 | `distance` | `double` | `Frame.D` | Distance since previous frame in meters. `0` when ball is dead. |
| 12 | `acceleration` | `double` | `Frame.A` | Acceleration in m/s². |
| 13 | `m_flag` | `int` | `Frame.M` | Manual correction flag — tracking data was manually adjusted. |
| 14 | `ball_possession` | `int` | `Frame.BallPossession` | `1` = home team, `2` = guest team. Only set on BALL rows. |
| 15 | `ball_status` | `int` | `Frame.BallStatus` | `1` = alive (in play), `0` = dead (stoppage). `0` implies `distance=0`. |

**Key relationships** (verified in validation cells below):
* Coordinates are **absolute** — teams switch ends at halftime, so X positions flip sign between halves
* `ball_possession` is always 1 or 2 on every BALL row (even when dead)
* `ball_status=0` implies `distance=0` (always true)
* `Frame.N` is not loaded — `frame_id` (from `n`) is the sole frame identifier
* \~23.7M rows across 7 matches, \~3.4M rows per match

In [0]:
# ── 1. Read all JSONL files from S3 ──
# timemajor directory has 7 JSONL files, one per match, ~1GB each

from pyspark.sql.functions import col, explode, to_timestamp

raw_df = spark.read.json(
    "s3://bundesliga-2022-2023-data/timemajor/*.jsonl",
    multiLine=False
)

print(f"Total JSONL lines (frames): {raw_df.count()}")
print(f"Top-level columns: {raw_df.columns}")
raw_df.printSchema()

In [0]:
# ── 2. Explode frames array and flatten into one row per entity per frame ──
# Each JSONL line = one frame with an array of entities (players, ball, referees)
# Explode the array so each row = one entity at one frame timestamp.
# Cast all string fields to their proper types for efficient querying.

from pyspark.sql.functions import col, explode, to_timestamp

positions_df = raw_df.select(
    col("n").alias("frame_id"),
    explode("frames").alias("frame_obj")
).select(
    # FrameSet metadata
    col("frame_id"),
    col("frame_obj.FrameSet.MatchId").alias("match_id"),
    col("frame_obj.FrameSet.GameSection").alias("game_section"),
    col("frame_obj.FrameSet.TeamId").alias("team_id"),
    col("frame_obj.FrameSet.PersonId").alias("person_id"),
    to_timestamp(col("frame_obj.Frame.T")).alias("timestamp"),
    col("frame_obj.Frame.X").cast("double").alias("x"),
    col("frame_obj.Frame.Y").cast("double").alias("y"),
    col("frame_obj.Frame.Z").cast("double").alias("z"),
    col("frame_obj.Frame.S").cast("double").alias("speed"),
    col("frame_obj.Frame.D").cast("double").alias("distance"),
    col("frame_obj.Frame.A").cast("double").alias("acceleration"),
    col("frame_obj.Frame.M").cast("int").alias("m_flag"),
    col("frame_obj.Frame.BallPossession").cast("int").alias("ball_possession"),
    col("frame_obj.Frame.BallStatus").cast("int").alias("ball_status"),
)

print(f"Schema ({len(positions_df.columns)} columns):")
positions_df.printSchema()
print(f"\nSample:")
display(positions_df.limit(10))

In [0]:
# ── 3. Save to Delta table ──
# This is a large dataset (~25M+ rows across 7 matches).
# Using overwrite + overwriteSchema for the initial load.

catalog = "`bundesliga-2022-2023`"
schema_name = f"{catalog}.batch"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

(
    positions_df.write
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.raw_positions")
)

# Summary
tbl = spark.table(f"{schema_name}.raw_positions")
print(f"Table: {schema_name}.raw_positions")
print(f"  Rows: {tbl.count()}")
print(f"  Columns: {len(tbl.columns)}")
print(f"\nRows per match:")
tbl.groupBy("match_id").count().orderBy("match_id").show()
print(f"\nEntity types (team_id values):")
tbl.groupBy("team_id").count().orderBy(col("count").desc()).show()

In [0]:
# ── 4. Validate: all JSONL fields captured in Delta table ──
# Compare raw JSONL schema fields vs Delta table columns

from pyspark.sql.functions import col

tbl = spark.table("`bundesliga-2022-2023`.batch.raw_positions")

# All fields in the raw JSONL schema
raw_fields = {
    "n (frame_id)": "long",
    "FrameSet.MatchId (match_id)": "string",
    "FrameSet.GameSection (game_section)": "string",
    "FrameSet.TeamId (team_id)": "string",
    "FrameSet.PersonId (person_id)": "string",
    "Frame.T (timestamp)": "timestamp",
    "Frame.X (x)": "double",
    "Frame.Y (y)": "double",
    "Frame.Z (z)": "double",
    "Frame.S (speed)": "double",
    "Frame.D (distance)": "double",
    "Frame.A (acceleration)": "double",
    "Frame.M (m_flag)": "int",
    "Frame.BallPossession (ball_possession)": "int",
    "Frame.BallStatus (ball_status)": "int",
}

delta_cols = set(tbl.columns)
print(f"Delta table columns: {len(delta_cols)}")
print(f"Raw JSONL fields: {len(raw_fields)}")

missing = []
for raw_name, raw_type in raw_fields.items():
    delta_name = raw_name.split("(")[-1].rstrip(") ") if "(" in raw_name else raw_name
    if delta_name in delta_cols:
        print(f"  {raw_name:<45} -> {delta_name:<20} ✅")
    else:
        print(f"  {raw_name:<45} -> {delta_name:<20} ❌ MISSING")
        missing.append(delta_name)

print(f"\n{'✅ ALL fields captured!' if not missing else f'⚠️ Missing: {missing}'}")

# Summary stats
print(f"\n=== Data summary ===")
print(f"  Total rows: {tbl.count()}")
print(f"  Matches: {tbl.select('match_id').distinct().count()}")
print(f"  Unique persons: {tbl.select('person_id').distinct().count()}")
print(f"  Frame range: {tbl.agg({'frame_id': 'min'}).collect()[0][0]} - {tbl.agg({'frame_id': 'max'}).collect()[0][0]}")
print(f"  Time range: {tbl.agg({'timestamp': 'min'}).collect()[0][0]} - {tbl.agg({'timestamp': 'max'}).collect()[0][0]}")
print(f"\n  Rows with ball_possession: {tbl.filter(col('ball_possession').isNotNull()).count()}")
print(f"  Rows with ball_status: {tbl.filter(col('ball_status').isNotNull()).count()}")
print(f"  Rows with z (height): {tbl.filter(col('z').isNotNull()).count()}")
print(f"  Ball rows: {tbl.filter(col('team_id') == 'BALL').count()}")

In [0]:
# ── 5. Ball possession and status analysis ──
# ball_possession: 1 = home team, 2 = guest team (only set on BALL rows)
# ball_status: 1 = alive (in play), 0 = dead (out of play / stoppage)

from pyspark.sql.functions import col, when, lag
from pyspark.sql.window import Window

positions = spark.table("`bundesliga-2022-2023`.batch.raw_positions")
ball = positions.filter(col("team_id") == "BALL")

# 5a. Cross-tab: ball_status vs ball_possession
print("=== ball_status vs ball_possession ===")
ball.groupBy("ball_status", "ball_possession").count().orderBy("ball_status", "ball_possession").show()

# 5b. Distribution
print("=== ball_status distribution ===")
ball.groupBy("ball_status").count().withColumn(
    "pct", col("count") / ball.count() * 100
).orderBy("ball_status").show()

# 5c. Speed stats by ball_status
print("=== Speed stats: ball_status=0 (dead) ===")
ball.filter(col("ball_status") == 0).summary("count", "min", "max", "mean").select("summary", "speed").show()

print("=== Speed stats: ball_status=1 (alive) ===")
ball.filter(col("ball_status") == 1).summary("count", "min", "max", "mean").select("summary", "speed").show()

# 5d. Example: all entities at one frame where ball_possession=1
ball_row = ball.filter(
    (col("match_id") == "DFL-MAT-J03WMX") &
    (col("ball_possession") == 1)
).limit(1).collect()[0]

frame_num = ball_row["frame_id"]
print(f"=== All entities at frame {frame_num} (ball_possession=1, home team) ===")
display(
    positions.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == frame_num)
    ).select(
        "team_id", "person_id", "x", "y", "z", "speed",
        "ball_possession", "ball_status"
    ).orderBy(col("team_id"), col("person_id"))
)

# 5e. Ball possession transitions for one match
print(f"\n=== Match DFL-MAT-J03WMX: first 15 ball_possession transitions ===")
w = Window.partitionBy("match_id").orderBy("frame_id")
transitions = ball.filter(col("match_id") == "DFL-MAT-J03WMX") \
    .withColumn("prev_poss", lag(col("ball_possession")).over(w)) \
    .withColumn("prev_status", lag(col("ball_status")).over(w)) \
    .filter(
        (col("ball_possession") != col("prev_poss")) |
        (col("ball_status") != col("prev_status"))
    ) \
    .select("frame_id", "timestamp", "ball_possession", "ball_status", "x", "y", "speed") \
    .orderBy("frame_id")

display(transitions.limit(15))

# 5f. Possession % by team per match
print("\n=== Possession % by team per match (ball alive only) ===")
match_info = spark.table("`bundesliga-2022-2023`.batch.match_info")
poss_pct = ball.filter(col("ball_status") == 1) \
    .groupBy("match_id", "ball_possession").count() \
    .groupBy("match_id").pivot("ball_possession").sum("count") \
    .withColumn("home_pct", col("1") / (col("1") + col("2")) * 100) \
    .withColumn("guest_pct", col("2") / (col("1") + col("2")) * 100)

display(
    poss_pct.join(match_info, "match_id")
    .select("match_id", "match_title", "home_team_id", "guest_team_id", "home_pct", "guest_pct")
    .orderBy("match_id")
)

In [0]:
# ── 6. Verify: ball_status=0 always means distance=0? ──
# Checks the relationship between ball_status and distance for ball rows.

from pyspark.sql.functions import col, count, min as spark_min, max as spark_max, avg

ball = spark.table("`bundesliga-2022-2023`.batch.raw_positions").filter(col("team_id") == "BALL")

# Cross-tab: ball_status vs whether distance is 0
print("=== ball_status vs distance=0 (ball rows) ===")
ball.groupBy(
    col("ball_status"),
    (col("distance") == 0).alias("distance_is_zero")
).count().orderBy("ball_status", "distance_is_zero").show()

# Distance stats by ball_status
print("=== distance stats by ball_status ===")
ball.groupBy("ball_status").agg(
    count("*").alias("count"),
    spark_min("distance").alias("min_dist"),
    spark_max("distance").alias("max_dist"),
    avg("distance").alias("avg_dist")
).orderBy("ball_status").show()

# ball_status when distance > 0
print("=== ball_status when distance > 0 ===")
ball.filter(col("distance") > 0).groupBy("ball_status").count().show()

# ball_status when distance = 0
print("=== ball_status when distance = 0 ===")
ball.filter(col("distance") == 0).groupBy("ball_status").count().show()

print("""
Conclusion:
  ball_status=0 (dead) → distance=0  ALWAYS TRUE (449,539 rows)
  distance>0           → ball_status=1 (alive)  ALWAYS TRUE (548,158 rows)
  ball_status=1 (alive) → distance=0  NOT always (4,947 exceptions: ball alive but stationary)
""")

In [0]:
# ── 7. Verify: ball_status=1 → ball_possession always 1 or 2? ──
# Checks that ball_possession is never null and always 1 or 2 when ball is alive.

from pyspark.sql.functions import col

ball = spark.table("`bundesliga-2022-2023`.batch.raw_positions").filter(col("team_id") == "BALL")

print("=== ball_status=1 (alive): ball_possession distribution ===")
ball.filter(col("ball_status") == 1).groupBy("ball_possession").count().orderBy("ball_possession").show()

print("=== ball_status=0 (dead): ball_possession distribution ===")
ball.filter(col("ball_status") == 0).groupBy("ball_possession").count().orderBy("ball_possession").show()

null_alive = ball.filter((col("ball_status") == 1) & col("ball_possession").isNull()).count()
null_dead = ball.filter((col("ball_status") == 0) & col("ball_possession").isNull()).count()
print(f"Null ball_possession when ball_status=1 (alive): {null_alive}")
print(f"Null ball_possession when ball_status=0 (dead):  {null_dead}")

print("""
Conclusion:
  ball_possession is ALWAYS 1 or 2 on every ball row — never null, no other values.
  This holds regardless of ball_status (alive or dead).
  1 = home team, 2 = guest team.
""")

In [0]:
# ── 8. Verify: are X/Y coordinates absolute or attack-oriented? ──
# If absolute: teams switch sides at halftime → player X positions flip between halves.
# If attack-oriented: X always points toward opponent's goal → positions stay same sign.
# Test: compare goalkeeper avg X in firstHalf vs secondHalf.

from pyspark.sql.functions import col, avg, count, min as spark_min, max as spark_max

positions = spark.table("`bundesliga-2022-2023`.batch.raw_positions")
match_id = "DFL-MAT-J03WMX"
home_team = "DFL-CLU-000008"
guest_team = "DFL-CLU-00000G"

# Frame ranges per half
print("=== Frame range by game_section ===")
positions.filter(col("match_id") == match_id) \
    .groupBy("game_section") \
    .agg(
        spark_min("frame_id").alias("min_frame"),
        spark_max("frame_id").alias("max_frame")
    ).orderBy("game_section").show()

# Average X/Y by team in first 500 frames of each half
print("=== Average X/Y by team and half (first 500 frames) ===")
players = positions.filter(
    (col("match_id") == match_id) &
    (col("team_id").isin(home_team, guest_team)) &
    (col("frame_id").between(10000, 10500) |
     col("frame_id").between(100000, 100500))
)
players.groupBy("game_section", "team_id").agg(
    avg("x").alias("avg_x"),
    avg("y").alias("avg_y"),
    count("*").alias("n")
).orderBy("game_section", "team_id").show()

# Ball at kickoff of each half
print("=== Ball at kickoff of each half ===")
positions.filter(
    (col("match_id") == match_id) &
    (col("team_id") == "BALL") &
    (col("frame_id").isin([10000, 10001, 10002, 100000, 100001, 100002]))
).select("game_section", "frame_id", "x", "y").orderBy("frame_id").show()

# Goalkeeper positions: every player's avg X in first 200 frames of each half
print("=== Home team (DFL-CLU-000008): player avg X per half ===")
for half, start in [("firstHalf", 10000), ("secondHalf", 100000)]:
    print(f"\n{half}:")
    positions.filter(
        (col("match_id") == match_id) &
        (col("team_id") == home_team) &
        (col("frame_id").between(start, start + 200))
    ).groupBy("person_id").agg(
        avg("x").alias("avg_x"),
        avg("y").alias("avg_y"),
        count("*").alias("n")
    ).orderBy("avg_x").show(15)

print("=== Guest team (DFL-CLU-00000G): player avg X per half ===")
for half, start in [("firstHalf", 10000), ("secondHalf", 100000)]:
    print(f"\n{half}:")
    positions.filter(
        (col("match_id") == match_id) &
        (col("team_id") == guest_team) &
        (col("frame_id").between(start, start + 200))
    ).groupBy("person_id").agg(
        avg("x").alias("avg_x"),
        avg("y").alias("avg_y"),
        count("*").alias("n")
    ).orderBy("avg_x").show(15)

print("""
Conclusion: coordinates are ABSOLUTE (fixed to the pitch, NOT attack-oriented).

  Home GK (DFL-OBJ-0002HE): 1st half avg_x = +46.25 → 2nd half avg_x = -30.85 (flipped ends)
  Guest GK (DFL-OBJ-0002DR): 1st half avg_x = -39.93 → 2nd half avg_x = +42.28 (flipped ends)

  Teams physically switch sides at halftime. The coordinate system is:
    X axis: goal-to-goal, approximately -52.5 to +52.5 (pitch length ~105m)
    Y axis: sideline-to-sideline, approximately -34 to +34 (pitch width ~68m)
    Center: (0, 0) at the midpoint

  For attacking-direction analysis, you must flip X for one team in one half.
""")